In [1]:
import pandas as pd
import numpy as np
import glob
import os
import re
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

# ================= 配置区 =================
STOCK_DATA_PATH = r"D:\work\trade\data_center\storage\market_data\stock_daily"
OUTPUT_PATH = r"D:\work\data2\train_data.csv"

# ================= 核心函数 =================

def calculate_market_sentiment(folder_path):
    """
    计算市场情绪指标（基于全市场股票）
    
    参数:
        folder_path: parquet文件所在文件夹路径
    
    返回:
        DataFrame: 每日市场统计数据
    """
    print("正在计算全市场情绪指标（基于所有股票）...")
    
    # 1. 获取文件夹下所有的 parquet 文件
    file_pattern = os.path.join(folder_path, "*.parquet")
    all_files = glob.glob(file_pattern)
    
    if not all_files:
        print("未找到任何 .parquet 文件，请检查路径。")
        return pd.DataFrame()
    
    print(f"共发现 {len(all_files)} 个全市场股票文件用于计算情绪指标")
    
    processed_dfs = []

    # 2. 遍历处理每个文件（全市场）
    for file_path in tqdm(all_files, desc="Processing market sentiment"):
        try:
            # 读取 Parquet
            df = pd.read_parquet(file_path)
            
            # 确保 Index 是 datetime 格式
            if not isinstance(df.index, pd.DatetimeIndex):
                df.index = pd.to_datetime(df.index)
            
            # 数据清洗：去除close为NaN的数据
            df = df.dropna(subset=['close'])
            
            if df.empty:
                continue

            # --- 计算所需的中间指标 ---
            # 1. 计算涨跌幅
            df['pct_chg'] = df['close'].pct_change()
            
            # 2. 判断涨跌停
            epsilon = 1e-4
            df['is_limit_up'] = (df['close'] >= (df['limit_up'] - epsilon)).astype(int)
            df['is_limit_down'] = (df['close'] <= (df['limit_down'] + epsilon)).astype(int)
            
            # 3. 标记这只股票当天是有效的
            df['is_valid'] = 1
            
            # 4. 只保留需要的列以节省内存
            mini_df = df[['is_limit_up', 'is_limit_down', 'pct_chg', 'is_valid']]
            
            processed_dfs.append(mini_df)
            
        except Exception as e:
            continue

    if not processed_dfs:
        print("没有有效的市场数据")
        return pd.DataFrame()
        
    # 3. 合并所有股票的轻量级数据
    all_stocks = pd.concat(processed_dfs)

    # 4. 按日期 Groupby 聚合
    daily_stats = all_stocks.groupby(all_stocks.index).agg({
        'is_limit_up': 'sum',
        'is_limit_down': 'sum',
        'is_valid': 'sum',     # 当天有多少只股票有数据
        'pct_chg': 'mean'      # 当天平均涨跌幅
    })

    # 5. 计算比例
    daily_stats['limit_up_ratio'] = daily_stats['is_limit_up'] / daily_stats['is_valid']
    daily_stats['limit_down_ratio'] = daily_stats['is_limit_down'] / daily_stats['is_valid']

    # 6. 整理最终的 DataFrame 格式
    final_df = daily_stats.rename(columns={
        'is_limit_up': 'limit_up_count',       # 涨停个数
        'is_limit_down': 'limit_down_count',   # 跌停个数
        'pct_chg': 'avg_pct_change',           # 当天平均涨跌幅
    })

    # 调整列顺序
    final_df = final_df[[
        'limit_up_count', 
        'limit_down_count', 
        'limit_up_ratio', 
        'limit_down_ratio', 
        'avg_pct_change'
    ]]
    
    print(f"市场情绪数据计算完成，时间范围: {final_df.index.min()} 至 {final_df.index.max()}")
    print(f"市场情绪数据形状: {final_df.shape}")
    
    return final_df

def calculate_technical_features(df):
    """
    计算个股技术特征和标签
    
    返回:
        添加了技术特征和标签的DataFrame
    """
    # --- 计算技术指标 ---
    df['ma5'] = df['close'].rolling(window=5).mean()
    df['ma_vol_5'] = df['volume'].rolling(window=5).mean()
    df['pre_close'] = df['close'].shift(1)
    
    # 1. 形态强弱类
    df['upper_shadow'] = (df['high'] - df[['open', 'close']].max(axis=1)) / df['close']
    df['body_size'] = abs(df['close'] - df['open']) / df['open']
    df['ma5_bias'] = (df['close'] - df['ma5']) / df['ma5']
    df['vol_ratio'] = df['volume'] / df['ma_vol_5'].replace(0, np.nan)
    
    # 2. 波动类
    df['high_low_ratio'] = (df['high'] - df['low']) / df['pre_close']

    # --- 计算标签和未来特征 ---
    # 获取未来数据
    next_close = df['close'].shift(-1)
    next_open = df['open'].shift(-1)
    next_high = df['high'].shift(-1)
    
    
    # 标签1: (收阳) AND (冲高 > 3%)
    cond_1 = next_close > next_open
    cond_2 = (next_high - next_open) / next_open > 0.03
    df['label'] = (cond_1 & cond_2).astype(int)
    
    # 标签2: 第二天开盘涨幅（新增特征）
    # 使用fillna(0)处理最后一天没有第二天数据的情况
    df['next_open_change'] = (next_open - df['close']) / df['close']
    
    return df

def apply_strategy_filter(df):
    """
    N字反转筛选 + 过热过滤
    """
    # 1. 基础计算
    epsilon = 1e-4
    df['is_limit_up'] = (df['close'] >= (df['limit_up'] - epsilon)).astype(int)
    # 过去60天涨停数（不含今天，shift(1)）
    df['recent_limit_ups_60'] = df['is_limit_up'].shift(1).rolling(window=60).sum()
    
    # 2. 辅助列 (L1=昨天涨停...)
    df['L1'] = df['is_limit_up'].shift(1) == 1
    df['L2'] = df['is_limit_up'].shift(2) == 1
    df['L3'] = df['is_limit_up'].shift(3) == 1
    
    df['open_T1'] = df['open'].shift(1)
    df['open_T2'] = df['open'].shift(2)
    df['open_T3'] = df['open'].shift(3)
    
    # 3. 筛选 Mask
    
    # 过热逻辑 (过去60天涨停 < 6)
    not_overheated = (df['recent_limit_ups_60'] < 6)
    
    # 场景 1: T-1涨停，T未涨停
    past_counts_1 = df['is_limit_up'].shift(2).rolling(6).sum()
    case_1 = (
        (df['L1']) & (~df['is_limit_up'].astype(bool)) &
        (past_counts_1 < 2) &
        (df['close'] >= df['open_T1'])
    )

    # 场景 2: T-2涨停，T-1, T未涨停
    past_counts_2 = df['is_limit_up'].shift(3).rolling(6).sum()
    case_2 = (
        (df['L2']) & (~df['L1']) & (~df['is_limit_up'].astype(bool)) &
        (past_counts_2 < 2) &
        (df['close'] >= df['open_T2'])
    )

    # 场景 3: T-3涨停，T-2, T-1, T未涨停
    past_counts_3 = df['is_limit_up'].shift(4).rolling(6).sum()
    case_3 = (
        (df['L3']) & (~df['L2']) & (~df['L1']) & (~df['is_limit_up'].astype(bool)) &
        (past_counts_3 < 2) &
        (df['close'] >= df['open_T3'])
    )

    # 综合筛选
    final_mask = (case_1 | case_2 | case_3) & not_overheated & (df['is_valid'] == 1)
    
    return df[final_mask].copy()

def process_train_data(folder_path, include_last_day=True):
    """
    主处理函数：直接从数据源生成 train_data
    
    参数:
        folder_path: 数据文件夹路径
        include_last_day: 是否包含最后一天的数据（即使没有第二天数据）
    
    返回:
        DataFrame: 最终的训练数据
    """
    # ========== 第一阶段：计算全市场情绪指标 ==========
    df_market = calculate_market_sentiment(folder_path)
    if df_market.empty:
        print("市场情绪数据计算失败，程序退出")
        return pd.DataFrame()
    
    market_cols = df_market.columns.tolist()
    print(f"市场情绪特征: {market_cols}")
    
    # ========== 第二阶段：筛选60/00开头股票并处理 ==========
    print("\n正在筛选并处理60/00开头的股票...")
    
    # 1. 获取所有parquet文件
    file_pattern = os.path.join(folder_path, "*.parquet")
    all_files = glob.glob(file_pattern)
    
    if not all_files:
        print("未找到任何 .parquet 文件")
        return pd.DataFrame()
    
    # 2. 筛选 00 或 60 开头的股票 (排除 300, 688 等)
    target_pattern = re.compile(r'^(00|60)\d+') 
    valid_files = []
    
    for f in all_files:
        filename = os.path.basename(f)
        code = filename.split('.')[0]
        if target_pattern.match(code):
            valid_files.append(f)
            
    print(f"符合 60/00 开头的股票文件数: {len(valid_files)}")
    
    # 3. 获取最新的市场数据日期
    latest_market_date = df_market.index.max()
    print(f"最新的市场数据日期: {latest_market_date}")
    
    # 4. 处理每只目标股票
    result_list = []
    
    for file_path in tqdm(valid_files, desc="Processing target stocks"):
        try:
            # 读取股票数据
            df = pd.read_parquet(file_path)
            
            # 基础清洗和准备
            if df.empty:
                continue
                
            if not isinstance(df.index, pd.DatetimeIndex):
                df.index = pd.to_datetime(df.index)
                
            df = df.sort_index()
            df = df.dropna(subset=['close'])
            df['is_valid'] = 1
            
            # A. 计算个股技术特征和标签
            df = calculate_technical_features(df)
            
            # B. 策略筛选
            selected_df = apply_strategy_filter(df)
            
            if not selected_df.empty:
                # C. 合并市场情绪数据
                combined_df = selected_df.join(df_market, how='left')
                
                # 补充股票代码信息
                stock_code = os.path.basename(file_path).split('.')[0]
                combined_df['code'] = stock_code
                combined_df['date'] = combined_df.index
                
                # D. 处理最后一天的数据
                if include_last_day:
                    # 找到最后一天的数据
                    last_day_mask = combined_df.index == latest_market_date
                    if last_day_mask.any():
                        # 最后一天的next_open_change设置为0
                        combined_df.loc[last_day_mask, 'next_open_change'] = 0
                        # 最后一天的label可能无法确定，设置为0
                        combined_df.loc[last_day_mask, 'label'] = 0
                
                # E. 提取需要的列
                # 个股特征列
                individual_features = [
                    'upper_shadow', 'body_size', 'ma5_bias', 
                    'vol_ratio', 'high_low_ratio'
                ]
                
                # 检查哪些特征列实际存在
                available_features = [col for col in individual_features if col in combined_df.columns]
                
                # 构建要保留的列列表
                cols_to_keep = ['date', 'code'] + available_features + market_cols + ['label', 'next_open_change']
                
                # 只保留实际存在的列
                existing_cols = [col for col in cols_to_keep if col in combined_df.columns]
                
                result_list.append(combined_df[existing_cols])
                
        except Exception as e:
            print(f"处理文件 {os.path.basename(file_path)} 时出错: {str(e)[:100]}")
            continue
    
    # ========== 第三阶段：汇总结果 ==========
    print("\n正在合并最终数据集...")
    if result_list:
        final_train_data = pd.concat(result_list, ignore_index=True)
        
        # 排序
        final_train_data = final_train_data.sort_values(by=['date', 'code'])
        
        # 处理缺失值
        # 1. 填充next_open_change为0（最后一天数据）
        final_train_data['next_open_change'] = final_train_data['next_open_change'].fillna(0)
        
        # 2. 填充label为0（最后一天数据）
        final_train_data['label'] = final_train_data['label'].fillna(0)
        
        # 3. 清理其他关键特征为NaN的数据
        # 只删除真正无效的数据，保留最后一天
        if include_last_day:
            # 不删除最后一天的数据
            date_mask = final_train_data['date'] < latest_market_date
            final_train_data_clean = final_train_data[~(
                date_mask & 
                (final_train_data['label'].isna() | 
                 final_train_data['next_open_change'].isna())
            )]
        else:
            # 删除所有包含NaN的行
            final_train_data_clean = final_train_data.dropna(
                subset=['label', 'next_open_change']
            )
        
        # 数据统计
        print(f"原始数据样本数: {len(final_train_data)}")
        print(f"清理后样本数: {len(final_train_data_clean)}")
        print(f"数据时间范围: {final_train_data_clean['date'].min()} 至 {final_train_data_clean['date'].max()}")
        
        # 检查最后一天是否有数据
        last_day_data = final_train_data_clean[final_train_data_clean['date'] == latest_market_date]
        if not last_day_data.empty:
            print(f"最后一天 ({latest_market_date.date()}) 样本数: {len(last_day_data)}")
        else:
            print(f"警告：最后一天 ({latest_market_date.date()}) 没有数据")
        
        # 检查是否有列缺失
        print(f"\n包含特征列 ({len(final_train_data_clean.columns)}个):")
        for i, col in enumerate(final_train_data_clean.columns.tolist(), 1):
            print(f"{i:2d}. {col}")
        
        # 标签分布统计
        print(f"\n标签分布:")
        label_counts = final_train_data_clean['label'].value_counts().sort_index()
        for label, count in label_counts.items():
            percentage = count / len(final_train_data_clean) * 100
            print(f"  label={label}: {count}个样本 ({percentage:.2f}%)")
        
        # 按日期统计样本数
        print(f"\n每日样本数统计:")
        daily_counts = final_train_data_clean['date'].value_counts().sort_index()
        print(f"  最早日期: {daily_counts.index.min()}，样本数: {daily_counts.iloc[0]}")
        print(f"  最晚日期: {daily_counts.index.max()}，样本数: {daily_counts.iloc[-1]}")
        print(f"  平均每日样本数: {daily_counts.mean():.1f}")
        
        # next_open_change统计
        if 'next_open_change' in final_train_data_clean.columns:
            print(f"\nnext_open_change统计:")
            print(f"  均值: {final_train_data_clean['next_open_change'].mean():.4f}")
            print(f"  标准差: {final_train_data_clean['next_open_change'].std():.4f}")
            print(f"  最小值: {final_train_data_clean['next_open_change'].min():.4f}")
            print(f"  最大值: {final_train_data_clean['next_open_change'].max():.4f}")
            print(f"  非零值比例: {(final_train_data_clean['next_open_change'] != 0).mean():.2%}")
        
        # 创建目录并保存
        os.makedirs(os.path.dirname(OUTPUT_PATH), exist_ok=True)
        final_train_data_clean.to_csv(OUTPUT_PATH, index=False)
        print(f"\n全部完成！文件保存至: {OUTPUT_PATH}")
        
        return final_train_data_clean
    else:
        print("未筛选出任何符合条件的样本。")
        return pd.DataFrame()

# ================= 主程序入口 =================
if __name__ == "__main__":
    print("开始生成训练数据...")
    print("=" * 50)
    
    # 包含最后一天的数据（即使没有第二天数据）
    train_data = process_train_data(STOCK_DATA_PATH, include_last_day=True)
    
    if not train_data.empty:
        print("\n" + "=" * 50)
        print("数据生成完成！")
        print(f"总样本数: {len(train_data)}")
        print(f"总特征数: {len(train_data.columns)}")
        print(f"数据时间范围: {train_data['date'].min()} 至 {train_data['date'].max()}")
        
        # 显示最后几行数据
        print("\n最后5天数据预览:")
        latest_dates = train_data['date'].unique()[-5:]
        for date in latest_dates:
            date_data = train_data[train_data['date'] == date]
            print(f"\n日期: {date.date()}，样本数: {len(date_data)}")
            if len(date_data) > 0:
                print(f"  next_open_change均值: {date_data['next_open_change'].mean():.4f}")
                print(f"  label分布: {dict(date_data['label'].value_counts())}")

开始生成训练数据...
正在计算全市场情绪指标（基于所有股票）...
共发现 5177 个全市场股票文件用于计算情绪指标


Processing market sentiment: 100%|██████████| 5177/5177 [01:15<00:00, 68.69it/s]


市场情绪数据计算完成，时间范围: 2014-01-02 00:00:00 至 2025-12-18 00:00:00
市场情绪数据形状: (2910, 5)
市场情绪特征: ['limit_up_count', 'limit_down_count', 'limit_up_ratio', 'limit_down_ratio', 'avg_pct_change']

正在筛选并处理60/00开头的股票...
符合 60/00 开头的股票文件数: 3188
最新的市场数据日期: 2025-12-18 00:00:00


Processing target stocks: 100%|██████████| 3188/3188 [00:49<00:00, 64.57it/s]



正在合并最终数据集...
原始数据样本数: 171056
清理后样本数: 171056
数据时间范围: 2014-04-03 00:00:00 至 2025-12-18 00:00:00
最后一天 (2025-12-18) 样本数: 63

包含特征列 (14个):
 1. date
 2. code
 3. upper_shadow
 4. body_size
 5. ma5_bias
 6. vol_ratio
 7. high_low_ratio
 8. limit_up_count
 9. limit_down_count
10. limit_up_ratio
11. limit_down_ratio
12. avg_pct_change
13. label
14. next_open_change

标签分布:
  label=0: 110656个样本 (64.69%)
  label=1: 60400个样本 (35.31%)

每日样本数统计:
  最早日期: 2014-04-03 00:00:00，样本数: 23
  最晚日期: 2025-12-18 00:00:00，样本数: 63
  平均每日样本数: 60.0

next_open_change统计:
  均值: -0.0065
  标准差: 0.0213
  最小值: -0.6780
  最大值: 0.1017
  非零值比例: 93.82%

全部完成！文件保存至: D:\work\data2\train_data.csv

数据生成完成！
总样本数: 171056
总特征数: 14
数据时间范围: 2014-04-03 00:00:00 至 2025-12-18 00:00:00

最后5天数据预览:

日期: 2025-12-12，样本数: 38
  next_open_change均值: -0.0138
  label分布: {0: 27, 1: 11}

日期: 2025-12-15，样本数: 61
  next_open_change均值: -0.0036
  label分布: {0: 47, 1: 14}

日期: 2025-12-16，样本数: 66
  next_open_change均值: -0.0045
  label分布: {0: 45, 1: 21}

日期: 202

In [5]:
import pandas as pd
import numpy as np
import glob
import os
import re
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

# ================= 配置区 =================
STOCK_DATA_PATH = r"D:\work\trade\data_center\storage\market_data\stock_daily"
OUTPUT_PATH = r"D:\work\data2\train_data_T+2_red.csv"

# ================= 核心函数 =================

def calculate_market_sentiment(folder_path):
    """
    计算市场情绪指标（基于全市场股票）
    
    参数:
        folder_path: parquet文件所在文件夹路径
    
    返回:
        DataFrame: 每日市场统计数据
    """
    print("正在计算全市场情绪指标（基于所有股票）...")
    
    # 1. 获取文件夹下所有的 parquet 文件
    file_pattern = os.path.join(folder_path, "*.parquet")
    all_files = glob.glob(file_pattern)
    
    if not all_files:
        print("未找到任何 .parquet 文件，请检查路径。")
        return pd.DataFrame()
    
    print(f"共发现 {len(all_files)} 个全市场股票文件用于计算情绪指标")
    
    processed_dfs = []

    # 2. 遍历处理每个文件（全市场）
    for file_path in tqdm(all_files, desc="Processing market sentiment"):
        try:
            # 读取 Parquet
            df = pd.read_parquet(file_path)
            
            # 确保 Index 是 datetime 格式
            if not isinstance(df.index, pd.DatetimeIndex):
                df.index = pd.to_datetime(df.index)
            
            # 数据清洗：去除close为NaN的数据
            df = df.dropna(subset=['close'])
            
            if df.empty:
                continue

            # --- 计算所需的中间指标 ---
            # 1. 计算涨跌幅
            df['pct_chg'] = df['close'].pct_change()
            
            # 2. 判断涨跌停
            epsilon = 1e-4
            df['is_limit_up'] = (df['close'] >= (df['limit_up'] - epsilon)).astype(int)
            df['is_limit_down'] = (df['close'] <= (df['limit_down'] + epsilon)).astype(int)
            
            # 3. 标记这只股票当天是有效的
            df['is_valid'] = 1
            
            # 4. 只保留需要的列以节省内存
            mini_df = df[['is_limit_up', 'is_limit_down', 'pct_chg', 'is_valid']]
            
            processed_dfs.append(mini_df)
            
        except Exception as e:
            continue

    if not processed_dfs:
        print("没有有效的市场数据")
        return pd.DataFrame()
        
    # 3. 合并所有股票的轻量级数据
    all_stocks = pd.concat(processed_dfs)

    # 4. 按日期 Groupby 聚合
    daily_stats = all_stocks.groupby(all_stocks.index).agg({
        'is_limit_up': 'sum',
        'is_limit_down': 'sum',
        'is_valid': 'sum',     # 当天有多少只股票有数据
        'pct_chg': 'mean'      # 当天平均涨跌幅
    })

    # 5. 计算比例
    daily_stats['limit_up_ratio'] = daily_stats['is_limit_up'] / daily_stats['is_valid']
    daily_stats['limit_down_ratio'] = daily_stats['is_limit_down'] / daily_stats['is_valid']

    # 6. 整理最终的 DataFrame 格式
    final_df = daily_stats.rename(columns={
        'is_limit_up': 'limit_up_count',       # 涨停个数
        'is_limit_down': 'limit_down_count',   # 跌停个数
        'pct_chg': 'avg_pct_change',           # 当天平均涨跌幅
    })

    # 调整列顺序
    final_df = final_df[[
        'limit_up_count', 
        'limit_down_count', 
        'limit_up_ratio', 
        'limit_down_ratio', 
        'avg_pct_change'
    ]]
    
    print(f"市场情绪数据计算完成，时间范围: {final_df.index.min()} 至 {final_df.index.max()}")
    print(f"市场情绪数据形状: {final_df.shape}")
    
    return final_df

def calculate_technical_features(df):
    """
    计算个股技术特征和标签
    
    返回:
        添加了技术特征和标签的DataFrame
    """
    # --- 计算技术指标 ---
    df['ma5'] = df['close'].rolling(window=5).mean()
    df['ma_vol_5'] = df['volume'].rolling(window=5).mean()
    df['pre_close'] = df['close'].shift(1)
    
    # 1. 形态强弱类
    df['upper_shadow'] = (df['high'] - df[['open', 'close']].max(axis=1)) / df['close']
    df['body_size'] = abs(df['close'] - df['open']) / df['open']
    df['ma5_bias'] = (df['close'] - df['ma5']) / df['ma5']
    df['vol_ratio'] = df['volume'] / df['ma_vol_5'].replace(0, np.nan)
    
    # 2. 波动类
    df['high_low_ratio'] = (df['high'] - df['low']) / df['pre_close']

    # --- 计算标签和未来特征 ---
    # 获取未来数据
    next_close = df['close'].shift(-1)
    next_open = df['open'].shift(-1)
    next_high = df['high'].shift(-1)
    
        # ⭐ 新增：获取 T+2 的 open
    next2_open = df['open'].shift(-2)
    # 标签1: (收阳) AND (冲高 > 3%)
    cond_1 = next_close > next_open
    cond_2 = (next_high - next_open) / next_open > 0.03
        # ⭐ 条件 C：T+2 开盘跌幅不超过 3%
    # 即 Open(T+2) ≥ 97% * Close(T+1)
    cond_3 = next2_open / next_close >= 1  

    # ⭐ 新 label：三个条件都满足 → 1，否则 0
    df['label'] = (cond_1 & cond_2 & cond_3).astype(int)

    
    # 标签2: 第二天开盘涨幅（新增特征）
    # 使用fillna(0)处理最后一天没有第二天数据的情况
    df['next_open_change'] = (next_open - df['close']) / df['close']
    
    return df

def apply_strategy_filter(df):
    """
    N字反转筛选 + 过热过滤
    """
    # 1. 基础计算
    epsilon = 1e-4
    df['is_limit_up'] = (df['close'] >= (df['limit_up'] - epsilon)).astype(int)
    # 过去60天涨停数（不含今天，shift(1)）
    df['recent_limit_ups_60'] = df['is_limit_up'].shift(1).rolling(window=60).sum()
    
    # 2. 辅助列 (L1=昨天涨停...)
    df['L1'] = df['is_limit_up'].shift(1) == 1
    df['L2'] = df['is_limit_up'].shift(2) == 1
    df['L3'] = df['is_limit_up'].shift(3) == 1
    
    df['open_T1'] = df['open'].shift(1)
    df['open_T2'] = df['open'].shift(2)
    df['open_T3'] = df['open'].shift(3)
    
    # 3. 筛选 Mask
    
    # 过热逻辑 (过去60天涨停 < 6)
    not_overheated = (df['recent_limit_ups_60'] < 6)
    
    # 场景 1: T-1涨停，T未涨停
    past_counts_1 = df['is_limit_up'].shift(2).rolling(6).sum()
    case_1 = (
        (df['L1']) & (~df['is_limit_up'].astype(bool)) &
        (past_counts_1 < 2) &
        (df['close'] >= df['open_T1'])
    )

    # 场景 2: T-2涨停，T-1, T未涨停
    past_counts_2 = df['is_limit_up'].shift(3).rolling(6).sum()
    case_2 = (
        (df['L2']) & (~df['L1']) & (~df['is_limit_up'].astype(bool)) &
        (past_counts_2 < 2) &
        (df['close'] >= df['open_T2'])
    )

    # 场景 3: T-3涨停，T-2, T-1, T未涨停
    past_counts_3 = df['is_limit_up'].shift(4).rolling(6).sum()
    case_3 = (
        (df['L3']) & (~df['L2']) & (~df['L1']) & (~df['is_limit_up'].astype(bool)) &
        (past_counts_3 < 2) &
        (df['close'] >= df['open_T3'])
    )

    # 综合筛选
    final_mask = (case_1 | case_2 | case_3) & not_overheated & (df['is_valid'] == 1)
    
    return df[final_mask].copy()

def process_train_data(folder_path, include_last_day=True):
    """
    主处理函数：直接从数据源生成 train_data
    
    参数:
        folder_path: 数据文件夹路径
        include_last_day: 是否包含最后一天的数据（即使没有第二天数据）
    
    返回:
        DataFrame: 最终的训练数据
    """
    # ========== 第一阶段：计算全市场情绪指标 ==========
    df_market = calculate_market_sentiment(folder_path)
    if df_market.empty:
        print("市场情绪数据计算失败，程序退出")
        return pd.DataFrame()
    
    market_cols = df_market.columns.tolist()
    print(f"市场情绪特征: {market_cols}")
    
    # ========== 第二阶段：筛选60/00开头股票并处理 ==========
    print("\n正在筛选并处理60/00开头的股票...")
    
    # 1. 获取所有parquet文件
    file_pattern = os.path.join(folder_path, "*.parquet")
    all_files = glob.glob(file_pattern)
    
    if not all_files:
        print("未找到任何 .parquet 文件")
        return pd.DataFrame()
    
    # 2. 筛选 00 或 60 开头的股票 (排除 300, 688 等)
    target_pattern = re.compile(r'^(00|60)\d+') 
    valid_files = []
    
    for f in all_files:
        filename = os.path.basename(f)
        code = filename.split('.')[0]
        if target_pattern.match(code):
            valid_files.append(f)
            
    print(f"符合 60/00 开头的股票文件数: {len(valid_files)}")
    
    # 3. 获取最新的市场数据日期
    latest_market_date = df_market.index.max()
    print(f"最新的市场数据日期: {latest_market_date}")
    
    # 4. 处理每只目标股票
    result_list = []
    
    for file_path in tqdm(valid_files, desc="Processing target stocks"):
        try:
            # 读取股票数据
            df = pd.read_parquet(file_path)
            
            # 基础清洗和准备
            if df.empty:
                continue
                
            if not isinstance(df.index, pd.DatetimeIndex):
                df.index = pd.to_datetime(df.index)
                
            df = df.sort_index()
            df = df.dropna(subset=['close'])
            df['is_valid'] = 1
            
            # A. 计算个股技术特征和标签
            df = calculate_technical_features(df)
            
            # B. 策略筛选
            selected_df = apply_strategy_filter(df)
            
            if not selected_df.empty:
                # C. 合并市场情绪数据
                combined_df = selected_df.join(df_market, how='left')
                
                # 补充股票代码信息
                stock_code = os.path.basename(file_path).split('.')[0]
                combined_df['code'] = stock_code
                combined_df['date'] = combined_df.index
                
                # D. 处理最后一天的数据
                if include_last_day:
                    # 找到最后一天的数据
                    last_day_mask = combined_df.index == latest_market_date
                    if last_day_mask.any():
                        # 最后一天的next_open_change设置为0
                        combined_df.loc[last_day_mask, 'next_open_change'] = 0
                        # 最后一天的label可能无法确定，设置为0
                        combined_df.loc[last_day_mask, 'label'] = 0
                
                # E. 提取需要的列
                # 个股特征列
                individual_features = [
                    'upper_shadow', 'body_size', 'ma5_bias', 
                    'vol_ratio', 'high_low_ratio'
                ]
                
                # 检查哪些特征列实际存在
                available_features = [col for col in individual_features if col in combined_df.columns]
                
                # 构建要保留的列列表
                cols_to_keep = ['date', 'code'] + available_features + market_cols + ['label', 'next_open_change']
                
                # 只保留实际存在的列
                existing_cols = [col for col in cols_to_keep if col in combined_df.columns]
                
                result_list.append(combined_df[existing_cols])
                
        except Exception as e:
            print(f"处理文件 {os.path.basename(file_path)} 时出错: {str(e)[:100]}")
            continue
    
    # ========== 第三阶段：汇总结果 ==========
    print("\n正在合并最终数据集...")
    if result_list:
        final_train_data = pd.concat(result_list, ignore_index=True)
        
        # 排序
        final_train_data = final_train_data.sort_values(by=['date', 'code'])
        
        # 处理缺失值
        # 1. 填充next_open_change为0（最后一天数据）
        final_train_data['next_open_change'] = final_train_data['next_open_change'].fillna(0)
        
        # 2. 填充label为0（最后一天数据）
        final_train_data['label'] = final_train_data['label'].fillna(0)
        
        # 3. 清理其他关键特征为NaN的数据
        # 只删除真正无效的数据，保留最后一天
        if include_last_day:
            # 不删除最后一天的数据
            date_mask = final_train_data['date'] < latest_market_date
            final_train_data_clean = final_train_data[~(
                date_mask & 
                (final_train_data['label'].isna() | 
                 final_train_data['next_open_change'].isna())
            )]
        else:
            # 删除所有包含NaN的行
            final_train_data_clean = final_train_data.dropna(
                subset=['label', 'next_open_change']
            )
        
        # 数据统计
        print(f"原始数据样本数: {len(final_train_data)}")
        print(f"清理后样本数: {len(final_train_data_clean)}")
        print(f"数据时间范围: {final_train_data_clean['date'].min()} 至 {final_train_data_clean['date'].max()}")
        
        # 检查最后一天是否有数据
        last_day_data = final_train_data_clean[final_train_data_clean['date'] == latest_market_date]
        if not last_day_data.empty:
            print(f"最后一天 ({latest_market_date.date()}) 样本数: {len(last_day_data)}")
        else:
            print(f"警告：最后一天 ({latest_market_date.date()}) 没有数据")
        
        # 检查是否有列缺失
        print(f"\n包含特征列 ({len(final_train_data_clean.columns)}个):")
        for i, col in enumerate(final_train_data_clean.columns.tolist(), 1):
            print(f"{i:2d}. {col}")
        
        # 标签分布统计
        print(f"\n标签分布:")
        label_counts = final_train_data_clean['label'].value_counts().sort_index()
        for label, count in label_counts.items():
            percentage = count / len(final_train_data_clean) * 100
            print(f"  label={label}: {count}个样本 ({percentage:.2f}%)")
        
        # 按日期统计样本数
        print(f"\n每日样本数统计:")
        daily_counts = final_train_data_clean['date'].value_counts().sort_index()
        print(f"  最早日期: {daily_counts.index.min()}，样本数: {daily_counts.iloc[0]}")
        print(f"  最晚日期: {daily_counts.index.max()}，样本数: {daily_counts.iloc[-1]}")
        print(f"  平均每日样本数: {daily_counts.mean():.1f}")
        
        # next_open_change统计
        if 'next_open_change' in final_train_data_clean.columns:
            print(f"\nnext_open_change统计:")
            print(f"  均值: {final_train_data_clean['next_open_change'].mean():.4f}")
            print(f"  标准差: {final_train_data_clean['next_open_change'].std():.4f}")
            print(f"  最小值: {final_train_data_clean['next_open_change'].min():.4f}")
            print(f"  最大值: {final_train_data_clean['next_open_change'].max():.4f}")
            print(f"  非零值比例: {(final_train_data_clean['next_open_change'] != 0).mean():.2%}")
        
        # 创建目录并保存
        os.makedirs(os.path.dirname(OUTPUT_PATH), exist_ok=True)
        final_train_data_clean.to_csv(OUTPUT_PATH, index=False)
        print(f"\n全部完成！文件保存至: {OUTPUT_PATH}")
        
        return final_train_data_clean
    else:
        print("未筛选出任何符合条件的样本。")
        return pd.DataFrame()

# ================= 主程序入口 =================
if __name__ == "__main__":
    print("开始生成训练数据...")
    print("=" * 50)
    
    # 包含最后一天的数据（即使没有第二天数据）
    train_data = process_train_data(STOCK_DATA_PATH, include_last_day=True)
    
    if not train_data.empty:
        print("\n" + "=" * 50)
        print("数据生成完成！")
        print(f"总样本数: {len(train_data)}")
        print(f"总特征数: {len(train_data.columns)}")
        print(f"数据时间范围: {train_data['date'].min()} 至 {train_data['date'].max()}")
        
        # 显示最后几行数据
        print("\n最后5天数据预览:")
        latest_dates = train_data['date'].unique()[-5:]
        for date in latest_dates:
            date_data = train_data[train_data['date'] == date]
            print(f"\n日期: {date.date()}，样本数: {len(date_data)}")
            if len(date_data) > 0:
                print(f"  next_open_change均值: {date_data['next_open_change'].mean():.4f}")
                print(f"  label分布: {dict(date_data['label'].value_counts())}")

开始生成训练数据...
正在计算全市场情绪指标（基于所有股票）...
共发现 5177 个全市场股票文件用于计算情绪指标


Processing market sentiment:   0%|          | 0/5177 [00:00<?, ?it/s]

Processing market sentiment: 100%|██████████| 5177/5177 [00:29<00:00, 173.61it/s]


市场情绪数据计算完成，时间范围: 2014-01-02 00:00:00 至 2025-12-18 00:00:00
市场情绪数据形状: (2910, 5)
市场情绪特征: ['limit_up_count', 'limit_down_count', 'limit_up_ratio', 'limit_down_ratio', 'avg_pct_change']

正在筛选并处理60/00开头的股票...
符合 60/00 开头的股票文件数: 3188
最新的市场数据日期: 2025-12-18 00:00:00


Processing target stocks: 100%|██████████| 3188/3188 [00:52<00:00, 60.73it/s]



正在合并最终数据集...
原始数据样本数: 171056
清理后样本数: 171056
数据时间范围: 2014-04-03 00:00:00 至 2025-12-18 00:00:00
最后一天 (2025-12-18) 样本数: 63

包含特征列 (14个):
 1. date
 2. code
 3. upper_shadow
 4. body_size
 5. ma5_bias
 6. vol_ratio
 7. high_low_ratio
 8. limit_up_count
 9. limit_down_count
10. limit_up_ratio
11. limit_down_ratio
12. avg_pct_change
13. label
14. next_open_change

标签分布:
  label=0: 148975个样本 (87.09%)
  label=1: 22081个样本 (12.91%)

每日样本数统计:
  最早日期: 2014-04-03 00:00:00，样本数: 23
  最晚日期: 2025-12-18 00:00:00，样本数: 63
  平均每日样本数: 60.0

next_open_change统计:
  均值: -0.0065
  标准差: 0.0213
  最小值: -0.6780
  最大值: 0.1017
  非零值比例: 93.82%

全部完成！文件保存至: D:\work\data2\train_data_T+2_red.csv

数据生成完成！
总样本数: 171056
总特征数: 14
数据时间范围: 2014-04-03 00:00:00 至 2025-12-18 00:00:00

最后5天数据预览:

日期: 2025-12-12，样本数: 38
  next_open_change均值: -0.0138
  label分布: {0: 32, 1: 6}

日期: 2025-12-15，样本数: 61
  next_open_change均值: -0.0036
  label分布: {0: 60, 1: 1}

日期: 2025-12-16，样本数: 66
  next_open_change均值: -0.0045
  label分布: {0: 60, 1: 6}

日期